# 05 · AI Video Investigator demo (Step 7)

Upload or pick a video, type what you are looking for, and get the top 5 moments as playable clips. The app runs **exactly the system from Step 4**: SigLIP frame evidence and X-CLIP activity evidence, fused with the α and proposal settings chosen on val (loaded from `vidsearch-fusion`).

Before launching, the notebook checks that the demo reproduces the benchmark: it re-indexes some Charades test videos from scratch and compares the results with Step 4's.

**Session settings:** Accelerator **GPU T4 x2** · Internet **On**
**Inputs (Add Input → Datasets):** `charades-sta-vidsearch`, `vidsearch-features`, `vidsearch-fusion`. Optional: `vidsearch-activity` (one extra check) and any dataset of surveillance videos such as UCF-Crime (shown as extra examples).

**Workflow:** click **Run All**. The last cell prints a public link to the app, which works while this session is running. **Do not commit this notebook**; stop the session when you are done.

In [ ]:
# 1. Get the code, run the tests, install Gradio
from pathlib import Path
import sys, glob, subprocess

REPO_URL = "https://github.com/zankhana46/AI-Video-Investigator.git"
!rm -rf /tmp/repo && git clone -q $REPO_URL /tmp/repo
sys.path.insert(0, "/tmp/repo")
for t in sorted(glob.glob("/tmp/repo/tests/test_*.py")):
    r = subprocess.run([sys.executable, t], capture_output=True, text=True, cwd="/tmp/repo")
    print(r.stdout.strip() or f"(no output from {Path(t).name})")
    if r.returncode:
        print(r.stderr[-2000:])
assert Path("/tmp/repo/vidsearch/demo.py").exists(), "Step 7 files are not on GitHub yet: push them first"

!pip install -q gradio

In [ ]:
# 2. Find the attached datasets
import json
from pathlib import Path
IN = Path("/kaggle/input")
print("attached:", sorted(p.name for p in IN.iterdir()))

ann = sorted(IN.rglob("test.jsonl"))
assert ann, "Attach charades-sta-vidsearch"
ANN_DIR = ann[0].parent
tars = [p for p in IN.rglob("videos.tar") if p.is_file()]
if tars:
    !tar -xf "{tars[0]}" -C /tmp
    VID_DIR = Path("/tmp/videos")
else:
    VID_DIR = next(p for p in IN.rglob("*.mp4") if "charades" in str(p).lower()).parent

siglip = [p for p in IN.rglob("*.npz") if not p.name.startswith(("xclip_", "scene_tags"))]
assert siglip, "Attach vidsearch-features"
FRAME_FEAT_DIR = siglip[0].parent

s4 = sorted(IN.rglob("step4_summary.json"))
pq = sorted(IN.rglob("per_query_top1_iou_step4.json"))
assert s4 and pq, "Attach vidsearch-fusion (Step 4 output)"
STEP4_SUMMARY, PQ_PATH = s4[0], pq[0]

meta = sorted(IN.rglob("xclip_meta.json"))
ACT_DIR = meta[0].parent if meta else None

print("videos:        ", VID_DIR, "| count:", len(list(VID_DIR.glob("*.mp4"))))
print("SigLIP cache:  ", FRAME_FEAT_DIR)
print("Step 4 config: ", STEP4_SUMMARY)
print("X-CLIP cache:  ", ACT_DIR or "not attached (optional)")

In [ ]:
# 3. Settings
import numpy as np
N_CHECK_VIDEOS = 15      # Charades test videos re-indexed from scratch to check the demo against Step 4
N_EXAMPLES = 6           # Charades examples offered in the app
MAX_MINUTES = 15         # longest video the app accepts
RES_OUT = Path("/kaggle/working/results")
RES_OUT.mkdir(parents=True, exist_ok=True)

In [ ]:
# 4. Load the final configuration from Step 4 and both models
from vidsearch import curves as cv, fusion as fu, scene_tags as st, demo
from vidsearch.xclip_features import XClipScorer

cfg = demo.load_final_config(STEP4_SUMMARY, demo.find_model_name(FRAME_FEAT_DIR))
for k, v in cfg.items():
    print(f"{k:18s} {v}")
fe = demo.FrameEncoder(cfg["frame_model"])
xs = XClipScorer(cfg["activity_model"])
searcher = demo.VideoSearch(cfg, fe, xs)
print("models loaded on", fe.device)

In [ ]:
# 5. Check A: do our fresh embeddings match the cached ones from notebooks 01 and 02?
test = cv.load_jsonl(ANN_DIR / "test.jsonl", "test")
PQ = json.load(open(PQ_PATH))
rng = np.random.default_rng(0)
check_vids = list(rng.choice(cv.videos_of(test), N_CHECK_VIDEOS, replace=False))
check_recs = [r for r in test if r["vid"] in set(check_vids)]

cached_txt = demo.load_qid_table(FRAME_FEAT_DIR, [r["qid"] for r in check_recs])
if cached_txt:
    qs = [r["query"] for r in check_recs if r["qid"] in cached_txt]
    ref = np.stack([cached_txt[r["qid"]] for r in check_recs if r["qid"] in cached_txt])
    ref /= np.linalg.norm(ref, axis=1, keepdims=True)
    best = None
    for mode in ("max_length", "longest"):           # match the text padding notebook 01 used
        fe.text_padding = mode
        c = float((fe.encode_text(qs) * ref).sum(1).mean())
        print(f"SigLIP text, padding={mode:10s}: mean cosine with notebook 01 = {c:.4f}")
        best = max(best or (c, mode), (c, mode))
    fe.text_padding = best[1]
    print("-> using padding =", fe.text_padding)

cached_frames = st.load_frame_table(FRAME_FEAT_DIR, check_vids[:3])
for v in check_vids[:3]:
    if cached_frames and v in cached_frames:
        e = fe.encode_frames(st.decode_1fps(VID_DIR / f"{v}.mp4")[0])
        c = cached_frames[v]; n = min(len(e), len(c))
        c = c[:n] / np.linalg.norm(c[:n], axis=1, keepdims=True)
        print(f"SigLIP frames, video {v}: mean cosine with notebook 01 = {float((e[:n] * c).sum(1).mean()):.4f}")

if ACT_DIR:
    from vidsearch.xclip_features import load_activity, load_video_clips
    act = load_activity(ACT_DIR)
    v = check_vids[0]
    clips = load_video_clips(VID_DIR / f"{v}.mp4", xs.frames_per_clip, cfg["clip_sec"], 1.0, xs.size)
    e, _ = xs.score_video(clips["frames"], clips["idx"], [])
    c = act["clip_emb"][v].astype(np.float32); n = min(len(e), len(c))
    cos = (e[:n] / np.linalg.norm(e[:n], axis=1, keepdims=True) * c[:n] / np.linalg.norm(c[:n], axis=1, keepdims=True)).sum(1)
    print(f"X-CLIP clips, video {v}: mean cosine with notebook 02 = {float(cos.mean()):.4f}")
print("(close to 1.0 = the demo encodes videos and queries exactly like the benchmark)")

In [ ]:
# 6. Check B: does the demo, re-indexing videos from scratch, reproduce Step 4's results?
import time
INDEX, demo_iou, bench_iou, q_ms = {}, [], [], []
for v in check_vids:
    INDEX[v] = searcher.index(VID_DIR / f"{v}.mp4")
for r in check_recs:
    t0 = time.time()
    s, a = searcher.curves(INDEX[r["vid"]], r["query"])
    segs, _, _ = demo.predict_segments(s, a, cfg, r.get("duration") or INDEX[r["vid"]]["duration"])
    q_ms.append(1000 * (time.time() - t0))
    demo_iou.append(float(fu.iou_array(np.array([segs[:1]]), np.array([[r["start"], r["end"]]]))[0, 0]))
    bench_iou.append(PQ["fused_hybrid"][r["qid"]])
demo_iou, bench_iou = np.array(demo_iou), np.array(bench_iou)
spm = np.mean([INDEX[v]["timing"]["s_per_video_minute"] for v in check_vids])
CHECK = {"queries": len(check_recs), "videos": len(check_vids),
         "demo_R1@0.5": round(100 * float((demo_iou >= 0.5).mean()), 1),
         "benchmark_R1@0.5": round(100 * float((bench_iou >= 0.5).mean()), 1),
         "same_hit_or_miss_pct": round(100 * float(((demo_iou >= 0.5) == (bench_iou >= 0.5)).mean()), 1),
         "mean_abs_iou_diff": round(float(np.abs(demo_iou - bench_iou).mean()), 3),
         "index_s_per_video_minute": round(float(spm), 2),
         "query_ms_median": round(float(np.median(q_ms)), 1)}
for k, val in CHECK.items():
    print(f"{k:26s} {val}")
json.dump(CHECK, open(RES_OUT / "demo_check.json", "w"), indent=2)

In [ ]:
# 7. Example videos for the app: Charades test videos where fusion fixed a query, plus any extra videos
fr, fused = PQ["frame_hybrid"], PQ["fused_hybrid"]
fixed = [r for r in test if fused[r["qid"]] >= 0.5 and fr[r["qid"]] < 0.3]
EXAMPLES, GT = [], {}
for r in fixed:
    if len(EXAMPLES) >= N_EXAMPLES or any(e["vid"] == r["vid"] for e in EXAMPLES):
        continue
    path = str(VID_DIR / f"{r['vid']}.mp4")
    qs = [r["query"]] + [x["query"] for x in test if x["vid"] == r["vid"] and x["qid"] != r["qid"]]
    EXAMPLES.append({"label": f"Charades {r['vid']}: {r['query']}", "path": path, "queries": qs, "vid": r["vid"]})
    for x in test:
        if x["vid"] == r["vid"]:
            GT[(path, x["query"].strip().lower())] = (x["start"], x["end"])
EXTRA = demo.find_extra_videos(IN, exclude_dir=VID_DIR, max_minutes=MAX_MINUTES)
EXAMPLES += EXTRA
print(f"{len(EXAMPLES) - len(EXTRA)} Charades examples, {len(EXTRA)} extra videos")
for e in EXAMPLES:
    print(" ", e["label"], "| try:", e["queries"][0])

In [ ]:
# 8. One search without the app (a figure for the report)
from IPython.display import display
ex = EXAMPLES[0]
idx = INDEX.get(ex.get("vid")) or searcher.index(ex["path"])
res = searcher.search(idx, ex["queries"][0], clips=False)
gt = GT.get((ex["path"], ex["queries"][0].strip().lower()))
fig = demo.plot_search(res["s"], res["a"], res["f"], res["segments"], res["scores"], ex["queries"][0], gt)
fig.savefig(RES_OUT / "demo_example.png", dpi=130)
display(fig)
for i, ((x0, x1), sc) in enumerate(zip(res["segments"], res["scores"])):
    print(f"#{i + 1}: {x0:5.1f} - {x1:5.1f} s   score {sc}")
print(f"query answered in {res['query_ms']} ms" + (f" | ground truth {gt[0]}-{gt[1]} s" if gt else ""))

In [ ]:
# 9. Launch the app (public link below; it works while this session runs)
app = demo.build_app(searcher, EXAMPLES, GT, max_minutes=MAX_MINUTES)
app.launch(share=True, allowed_paths=[str(searcher.clip_dir)], debug=False)

### Using the demo
1. Open the **public link** printed above (it ends in `gradio.live`). You can share it with your teammate; it stays valid while this session runs.
2. Pick an example (or upload a video up to 15 minutes), click **1. Index video**, then type a query and press **Search**.
3. For Charades examples, try the suggested query: the plot shows the ground truth in green and the app reports the IoU of the top moment.
4. Take screenshots for the report, then **stop the session** (power icon) so it stops using GPU quota.